# R02. miRNA–mRNA Integration Reanalysis

This reviewer-response notebook rebuilds the miRNA–mRNA integration using the **revised differential-expression results**.

## Analysis sets

- **Confirmatory miRNA:** BH-FDR significant miRNA(s)
- **Exploratory miRNAs:** moderated-test-only audit candidates
- **Exploratory mRNAs:** moderated-test-only audit candidates

## Databases

- miRDB v6.0
- TargetScanHuman v8.0
- miRTarBase v9.0 *(optional until the official file is available)*

The output is interpreted as **database-supported exploratory association**, not direct experimental validation in these cord-blood samples.


In [7]:
from pathlib import Path
import sys
import gzip
import zipfile
import re
import pandas as pd
import numpy as np
from IPython.display import display

cwd = Path.cwd().resolve()
if cwd.name == "revision":
    PROJECT_ROOT = cwd.parents[1]
elif cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RESULTS_DIR = PROJECT_ROOT / "results"
REVISION_DIR = RESULTS_DIR / "revision"
REVISION_DIR.mkdir(parents=True, exist_ok=True)

REF_DIR = PROJECT_ROOT / "data" / "reference" / "mirna_targets"
MIRDB_FILE = REF_DIR / "miRDB_v6" / "miRDB_v6.0_prediction_result.txt.gz"
TARGETSCAN_DIR = REF_DIR / "TargetScan_v8"
MIRTARBASE_FILE = REF_DIR / "miRTarBase_v9" / "miRTarBase_MTI.xlsx"

DEM_FILE = RESULTS_DIR / "DEM_moderated_only_candidates.csv"
DEG_FILE = RESULTS_DIR / "DEG_moderated_only_candidates.csv"
FDR_MIRNA_FILE = RESULTS_DIR / "miRNA_FDR_significant.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("REVISION_DIR =", REVISION_DIR)


PROJECT_ROOT = /Users/jihopark/Desktop/MCDA_revision_final
REVISION_DIR = /Users/jihopark/Desktop/MCDA_revision_final/results/revision


## 1. Check required files

In [8]:
required = [DEM_FILE, DEG_FILE, FDR_MIRNA_FILE, MIRDB_FILE]

for p in required:
    print(("FOUND" if p.exists() else "MISSING"), "->", p)

if not DEM_FILE.exists() or not DEG_FILE.exists() or not FDR_MIRNA_FILE.exists():
    raise FileNotFoundError("Run the revised differential-expression analysis first.")

if not MIRDB_FILE.exists():
    raise FileNotFoundError("miRDB v6.0 reference file is missing.")

targetscan_files = list(TARGETSCAN_DIR.glob("*"))
print("\nTargetScan files:")
for p in targetscan_files:
    print(" -", p.name)

print("\nmiRTarBase:",
      "FOUND" if MIRTARBASE_FILE.exists() else "OPTIONAL / NOT YET PROVIDED")


FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/DEM_moderated_only_candidates.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/DEG_moderated_only_candidates.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/miRNA_FDR_significant.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/miRDB_v6/miRDB_v6.0_prediction_result.txt.gz

TargetScan files:
 - Summary_Counts.default_predictions.txt.zip
 - miR_Family_Info.txt.zip

miRTarBase: FOUND


## 2. Load revised candidate sets

In [9]:
dem = pd.read_csv(DEM_FILE)
deg = pd.read_csv(DEG_FILE)
fdr_mirna = pd.read_csv(FDR_MIRNA_FILE)

exploratory_mirnas = set(dem["miRNA"].dropna().astype(str).str.strip())
exploratory_genes = set(deg["Gene"].dropna().astype(str).str.strip())
confirmatory_mirnas = set(fdr_mirna["miRNA"].dropna().astype(str).str.strip())

print("Exploratory miRNAs:", len(exploratory_mirnas))
print("Exploratory genes :", len(exploratory_genes))
print("Confirmatory miRNAs:", confirmatory_mirnas)


Exploratory miRNAs: 30
Exploratory genes : 114
Confirmatory miRNAs: {'hsa-miR-1292-5p'}


## 3. Parse miRDB v6.0

In [10]:
def parse_mirdb(path):
    # miRDB v6 download is tab-delimited and usually has no header.
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        sample = [next(f) for _ in range(5)]

    print("miRDB preview:")
    for line in sample:
        print(line.rstrip())

    # Try common schema: miRNA, target, score.
    df = pd.read_csv(
        path,
        sep="\\t",
        compression="gzip",
        header=None,
        comment="#",
        dtype=str
    )

    if df.shape[1] < 2:
        raise RuntimeError("Unexpected miRDB v6 format.")

    names = ["miRNA", "Gene"]
    if df.shape[1] >= 3:
        names.append("miRDB_score")
    names += [f"extra_{i}" for i in range(len(names), df.shape[1])]
    df.columns = names

    df["miRNA"] = df["miRNA"].astype(str).str.strip()
    df["Gene"] = df["Gene"].astype(str).str.strip()

    keep = df[
        df["miRNA"].isin(exploratory_mirnas) &
        df["Gene"].isin(exploratory_genes)
    ].copy()

    keep["miRDB"] = True
    return keep

mirdb_pairs = parse_mirdb(MIRDB_FILE)
print("\\nmiRDB revised candidate pairs:", len(mirdb_pairs))
display(mirdb_pairs.head())


miRDB preview:
cfa-miR-1185	XM_537211	59.3438099752
cfa-miR-1185	XM_536047	54.527
cfa-miR-1185	XM_005617022	55.1716326075
cfa-miR-1185	XM_014117861	57.4409058608
cfa-miR-1185	XM_014107884	57.1519


/var/folders/6_/gddy_by17qd62d0cvjbnjjz80000gn/T/ipykernel_32916/2267241434.py:11: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(


\nmiRDB revised candidate pairs: 0


,miRNA,Gene,miRDB_score,miRDB


## 4. Parse TargetScanHuman v8.0

In [11]:
def locate_targetscan_files(folder):
    summary = None
    mirfam = None

    for p in folder.glob("*"):
        n = p.name.lower()
        if "summary_counts" in n:
            summary = p
        elif "mir" in n and "family" in n:
            mirfam = p

    return summary, mirfam

summary_file, mirfam_file = locate_targetscan_files(TARGETSCAN_DIR)

print("Summary Counts:", summary_file)
print("miR Family    :", mirfam_file)

if summary_file is None:
    raise FileNotFoundError("TargetScan Summary Counts file not found.")

def read_table_auto(path):
    if path.suffix == ".zip":
        return pd.read_csv(path, sep="\t", compression="zip", low_memory=False)
    return pd.read_csv(path, sep="\t", low_memory=False)

ts = read_table_auto(summary_file)

print("\\nTargetScan columns:")
print(ts.columns.tolist())

# Identify useful columns robustly.
def find_col(df, patterns):
    for c in df.columns:
        low = str(c).lower()
        if all(p.lower() in low for p in patterns):
            return c
    return None

gene_col = (
    find_col(ts, ["gene", "symbol"])
    or find_col(ts, ["gene symbol"])
)

mirfam_col = (
    find_col(ts, ["mirna", "family"])
    or find_col(ts, ["mir family"])
)

rep_mir_col = (
    find_col(ts, ["representative", "mirna"])
    or find_col(ts, ["representative mirna"])
)

species_col = find_col(ts, ["species", "id"])

if gene_col is None:
    raise RuntimeError("Could not identify Gene Symbol column in TargetScan Summary Counts.")

# Human Species ID in TargetScan is normally 9606.
if species_col is not None:
    ts = ts[ts[species_col].astype(str) == "9606"].copy()

# Prefer representative miRNA if available; otherwise use family mapping later.
if rep_mir_col is not None:
    ts["miRNA"] = ts[rep_mir_col].astype(str).str.strip()
else:
    ts["miRNA"] = ""

ts["Gene"] = ts[gene_col].astype(str).str.strip()

# If direct representative-miRNA matching works, use it.
ts_direct = ts[
    ts["miRNA"].isin(exploratory_mirnas) &
    ts["Gene"].isin(exploratory_genes)
].copy()

print("Direct representative-miRNA matches:", len(ts_direct))

# If needed, map family -> mature miRNA using the miR Family file.
ts_pairs = ts_direct.copy()

if len(ts_pairs) == 0 and mirfam_file is not None and mirfam_col is not None:
    mf = read_table_auto(mirfam_file)
    print("\\nmiR Family columns:")
    print(mf.columns.tolist())

    mf_family = find_col(mf, ["mir", "family"]) or find_col(mf, ["family"])
    mf_mirbase = find_col(mf, ["mirbase", "id"])
    mf_species = find_col(mf, ["species", "id"])

    if mf_family and mf_mirbase:
        if mf_species:
            mf = mf[mf[mf_species].astype(str) == "9606"].copy()

        mf["miRNA"] = mf[mf_mirbase].astype(str).str.strip()
        mf["family_key"] = mf[mf_family].astype(str).str.strip()

        relevant_mf = mf[mf["miRNA"].isin(exploratory_mirnas)][
            ["miRNA", "family_key"]
        ].drop_duplicates()

        temp = ts.copy()
        temp["family_key"] = temp[mirfam_col].astype(str).str.strip()

        ts_pairs = temp.merge(
            relevant_mf,
            on="family_key",
            how="inner"
        )

        ts_pairs = ts_pairs[
            ts_pairs["Gene"].isin(exploratory_genes)
        ].copy()

ts_pairs["TargetScan"] = True

print("TargetScan revised candidate pairs:", len(ts_pairs))
display(ts_pairs[["miRNA", "Gene"]].drop_duplicates().head())


Summary Counts: /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/TargetScan_v8/Summary_Counts.default_predictions.txt.zip
miR Family    : /Users/jihopark/Desktop/MCDA_revision_final/data/reference/mirna_targets/TargetScan_v8/miR_Family_Info.txt.zip
\nTargetScan columns:
['Transcript ID', 'Gene Symbol', 'miRNA family', 'Species ID', 'Total num conserved sites', 'Number of conserved 8mer sites', 'Number of conserved 7mer-m8 sites', 'Number of conserved 7mer-1a sites', 'Total num nonconserved sites', 'Number of nonconserved 8mer sites', 'Number of nonconserved 7mer-m8 sites', 'Number of nonconserved 7mer-1a sites', 'Number of 6mer sites', 'Representative miRNA', 'Total context++ score', 'Cumulative weighted context++ score', 'Aggregate PCT', 'Predicted occupancy - low miRNA', 'Predicted occupancy - high miRNA', 'Predicted occupancy - transfected miRNA']
Direct representative-miRNA matches: 8
TargetScan revised candidate pairs: 8


,miRNA,Gene
7707,hsa-miR-125a-5p,SBNO1
40699,hsa-miR-125a-5p,GK5
46696,hsa-miR-128-3p,PRKX
139326,hsa-miR-128-3p,EIF5
193199,hsa-miR-128-3p,MOB1B


## 5. Parse miRTarBase v9.0 if available

In [12]:
def parse_mirtarbase(path):
    if not path.exists():
        return pd.DataFrame(columns=[
            "miRNA", "Gene", "miRTarBase",
            "Experiments", "Support Type", "PMID"
        ])

    df = pd.read_excel(path)

    print("miRTarBase columns:")
    print(df.columns.tolist())

    def find_any(candidates):
        for cand in candidates:
            for col in df.columns:
                if cand.lower() == str(col).strip().lower():
                    return col
        for cand in candidates:
            for col in df.columns:
                if cand.lower() in str(col).lower():
                    return col
        return None

    mir_col = find_any(["miRNA", "miRNA name"])
    gene_col = find_any(["Target Gene", "Target Gene Symbol", "Gene"])
    exp_col = find_any(["Experiments", "Experiment"])
    sup_col = find_any(["Support Type", "Support"])
    pmid_col = find_any(["References (PMID)", "PMID"])

    if mir_col is None or gene_col is None:
        raise RuntimeError("Could not identify miRNA / target-gene columns in miRTarBase.")

    out = pd.DataFrame({
        "miRNA": df[mir_col].astype(str).str.strip(),
        "Gene": df[gene_col].astype(str).str.strip(),
    })

    out["Experiments"] = df[exp_col] if exp_col else np.nan
    out["Support Type"] = df[sup_col] if sup_col else np.nan
    out["PMID"] = df[pmid_col] if pmid_col else np.nan

    out = out[
        out["miRNA"].isin(exploratory_mirnas) &
        out["Gene"].isin(exploratory_genes)
    ].copy()

    out["miRTarBase"] = True
    return out

mirtar_pairs = parse_mirtarbase(MIRTARBASE_FILE)

print("miRTarBase revised candidate pairs:", len(mirtar_pairs))
display(mirtar_pairs.head())


miRTarBase columns:
['miRTarBase ID', 'miRNA', 'Species (miRNA)', 'Target Gene', 'Target Gene (Entrez ID)', 'Species (Target Gene)', 'Experiments', 'Support Type', 'References (PMID)']
miRTarBase revised candidate pairs: 42


,miRNA,Gene,Experiments,Support Type,PMID,miRTarBase
15570,hsa-miR-148b-3p,TEX13A,Microarray,Functional MTI (Weak),17612493,True
18692,hsa-miR-128-3p,BLOC1S2,Sequencing,Functional MTI (Weak),20371350,True
18779,hsa-miR-128-3p,MOB1B,Microarray,Functional MTI (Weak),17612493,True
39343,hsa-miR-328-3p,HIST1H4D,CLASH,Functional MTI (Weak),23622248,True
41315,hsa-miR-125a-5p,PRC1,CLASH,Functional MTI (Weak),23622248,True


## 6. Integrate database support

In [13]:
def standardize_pairs(df, db):
    if df.empty:
        return pd.DataFrame(columns=["miRNA", "Gene", db])

    z = df[["miRNA", "Gene"]].drop_duplicates().copy()
    z[db] = True
    return z

a = standardize_pairs(mirdb_pairs, "miRDB")
b = standardize_pairs(ts_pairs, "TargetScan")
c = standardize_pairs(mirtar_pairs, "miRTarBase")

pairs = a.merge(b, on=["miRNA", "Gene"], how="outer")
pairs = pairs.merge(c, on=["miRNA", "Gene"], how="outer")

for col in ["miRDB", "TargetScan", "miRTarBase"]:
    if col not in pairs.columns:
        pairs[col] = False
    pairs[col] = pairs[col].fillna(False).astype(bool)

pairs["n_databases"] = (
    pairs[["miRDB", "TargetScan", "miRTarBase"]]
    .sum(axis=1)
)

pairs["confirmatory_miRNA"] = pairs["miRNA"].isin(confirmatory_mirnas)

print("Unique supported pairs:", len(pairs))
print(pairs["n_databases"].value_counts().sort_index())

display(
    pairs.sort_values(
        ["confirmatory_miRNA", "n_databases"],
        ascending=[False, False]
    ).head(30)
)


Unique supported pairs: 26
n_databases
1    23
2     3
Name: count, dtype: int64


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,confirmatory_miRNA
5,hsa-miR-128-3p,EIF5,False,True,True,2,False
6,hsa-miR-128-3p,MOB1B,False,True,True,2,False
13,hsa-miR-328-3p,HIST1H4D,False,True,True,2,False
0,hsa-miR-125a-5p,GK5,False,True,False,1,False
1,hsa-miR-125a-5p,PRC1,False,False,True,1,False
2,hsa-miR-125a-5p,SBNO1,False,True,False,1,False
3,hsa-miR-128-3p,BLOC1S2,False,False,True,1,False
4,hsa-miR-128-3p,CISD1,False,False,True,1,False
7,hsa-miR-128-3p,PRKX,False,True,False,1,False
8,hsa-miR-128-3p,SBNO1,False,True,False,1,False


## 7. Add revised expression direction

In [14]:
mi_stats = pd.read_csv(RESULTS_DIR / "miRNA_limma_full_results.csv")[
    ["miRNA", "logFC", "adj.P.Val"]
].rename(
    columns={
        "logFC": "miRNA_logFC",
        "adj.P.Val": "miRNA_FDR"
    }
)

mr_stats = pd.read_csv(RESULTS_DIR / "mRNA_limma_full_results.csv")[
    ["Gene", "logFC", "adj.P.Val"]
].rename(
    columns={
        "logFC": "mRNA_logFC",
        "adj.P.Val": "mRNA_FDR"
    }
)

pairs = pairs.merge(mi_stats, on="miRNA", how="left")
pairs = pairs.merge(mr_stats, on="Gene", how="left")

pairs["expression_relationship"] = np.where(
    np.sign(pairs["miRNA_logFC"]) != np.sign(pairs["mRNA_logFC"]),
    "anti-correlated",
    "same-direction"
)

pairs = pairs.sort_values(
    ["confirmatory_miRNA", "n_databases", "expression_relationship"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(pairs.head(50))


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,confirmatory_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship
0,hsa-miR-128-3p,EIF5,False,True,True,2,False,-1.174363,0.069791,-0.608970,0.965865,same-direction
1,hsa-miR-128-3p,MOB1B,False,True,True,2,False,-1.174363,0.069791,-0.818650,0.965865,same-direction
2,hsa-miR-328-3p,HIST1H4D,False,True,True,2,False,-1.094243,0.129738,-1.003720,0.965865,same-direction
3,hsa-miR-128-3p,CISD1,False,False,True,1,False,-1.174363,0.069791,0.616583,0.965865,anti-correlated
4,hsa-miR-148b-3p,TEX13A,False,False,True,1,False,-0.908403,0.109901,0.656873,0.965865,anti-correlated
5,hsa-miR-6779-5p,CBY3,False,False,True,1,False,-0.681383,0.178395,0.601470,0.965865,anti-correlated
6,hsa-miR-125a-5p,GK5,False,True,False,1,False,-0.729913,0.510138,-0.602703,0.965865,same-direction
7,hsa-miR-125a-5p,PRC1,False,False,True,1,False,-0.729913,0.510138,-0.587820,0.965865,same-direction
8,hsa-miR-125a-5p,SBNO1,False,True,False,1,False,-0.729913,0.510138,-0.588660,0.965865,same-direction
9,hsa-miR-128-3p,BLOC1S2,False,False,True,1,False,-1.174363,0.069791,-0.603343,0.965865,same-direction


## 8. Key subsets

In [15]:
confirmatory_pairs = pairs[
    pairs["confirmatory_miRNA"]
].copy()

anti_correlated = pairs[
    pairs["expression_relationship"] == "anti-correlated"
].copy()

multi_db = pairs[
    pairs["n_databases"] >= 2
].copy()

print("Pairs involving FDR-significant miRNA:", len(confirmatory_pairs))
print("Anti-correlated pairs:", len(anti_correlated))
print("Supported by >=2 databases:", len(multi_db))

print("\nConfirmatory-miRNA pairs:")
display(confirmatory_pairs.head(50))

print("\nPreviously highlighted genes:")
display(
    pairs[pairs["Gene"].isin(["CISD1", "TOMM40L", "PHC1", "SGOL1", "PRKX"])]
    .sort_values(["Gene", "n_databases"], ascending=[True, False])
)


Pairs involving FDR-significant miRNA: 0
Anti-correlated pairs: 3
Supported by >=2 databases: 3

Confirmatory-miRNA pairs:


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,confirmatory_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship



Previously highlighted genes:


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,confirmatory_miRNA,miRNA_logFC,miRNA_FDR,mRNA_logFC,mRNA_FDR,expression_relationship
3,hsa-miR-128-3p,CISD1,False,False,True,1,False,-1.174363,0.069791,0.616583,0.965865,anti-correlated
22,hsa-miR-6779-5p,PHC1,False,False,True,1,False,-0.681383,0.178395,-0.682983,0.965865,same-direction
10,hsa-miR-128-3p,PRKX,False,True,False,1,False,-1.174363,0.069791,-0.587287,0.965865,same-direction
12,hsa-miR-128-3p,TOMM40L,False,False,True,1,False,-1.174363,0.069791,-0.703597,0.965865,same-direction


## 9. Export R02 results

In [16]:
all_out = REVISION_DIR / "R02_all_supported_pairs.csv"
confirm_out = REVISION_DIR / "R02_FDR_miRNA_pairs.csv"
anti_out = REVISION_DIR / "R02_anti_correlated_pairs.csv"
multi_out = REVISION_DIR / "R02_multi_database_pairs.csv"
xlsx_out = REVISION_DIR / "R02_mirna_mrna_reanalysis.xlsx"

pairs.to_csv(all_out, index=False)
confirmatory_pairs.to_csv(confirm_out, index=False)
anti_correlated.to_csv(anti_out, index=False)
multi_db.to_csv(multi_out, index=False)

with pd.ExcelWriter(xlsx_out) as writer:
    pairs.to_excel(writer, sheet_name="All supported pairs", index=False)
    confirmatory_pairs.to_excel(writer, sheet_name="FDR miRNA pairs", index=False)
    anti_correlated.to_excel(writer, sheet_name="Anti-correlated", index=False)
    multi_db.to_excel(writer, sheet_name="Multi-database", index=False)

print("Saved R02 outputs to:")
print(REVISION_DIR)


Saved R02 outputs to:
/Users/jihopark/Desktop/MCDA_revision_final/results/revision


In [18]:
multi_db[
    [
        "miRNA",
        "Gene",
        "miRDB",
        "TargetScan",
        "miRTarBase",
        "n_databases",
        "miRNA_logFC",
        "mRNA_logFC",
        "expression_relationship",
    ]
].sort_values(
    "n_databases",
    ascending=False
)


,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,miRNA_logFC,mRNA_logFC,expression_relationship
0,hsa-miR-128-3p,EIF5,False,True,True,2,-1.174363,-0.60897,same-direction
1,hsa-miR-128-3p,MOB1B,False,True,True,2,-1.174363,-0.81865,same-direction
2,hsa-miR-328-3p,HIST1H4D,False,True,True,2,-1.094243,-1.00372,same-direction


In [19]:
anti_correlated[
    [
        "miRNA",
        "Gene",
        "miRDB",
        "TargetScan",
        "miRTarBase",
        "n_databases",
        "miRNA_logFC",
        "mRNA_logFC",
        "expression_relationship",
    ]
].sort_values(
    "n_databases",
    ascending=False
)

,miRNA,Gene,miRDB,TargetScan,miRTarBase,n_databases,miRNA_logFC,mRNA_logFC,expression_relationship
3,hsa-miR-128-3p,CISD1,False,False,True,1,-1.174363,0.616583,anti-correlated
4,hsa-miR-148b-3p,TEX13A,False,False,True,1,-0.908403,0.656873,anti-correlated
5,hsa-miR-6779-5p,CBY3,False,False,True,1,-0.681383,0.601470,anti-correlated


## R02 Summary

Integration of the moderated-test-only audit miRNA (n = 30) and mRNA
(n = 114) candidate sets identified two distinct classes of
database-supported miRNA–mRNA associations.

Three pairs were supported by at least two databases
(TargetScan and miRTarBase):

- hsa-miR-128-3p – EIF5
- hsa-miR-128-3p – MOB1B
- hsa-miR-328-3p – HIST1H4D

However, all three pairs showed same-direction expression changes
between T1 and T2.

Three additional pairs showed anti-correlated miRNA–mRNA expression:

- hsa-miR-128-3p – CISD1
- hsa-miR-148b-3p – TEX13A
- hsa-miR-6779-5p – CBY3

Each of these anti-correlated pairs was supported by miRTarBase
alone. No pair simultaneously showed multi-database support and
anti-correlated expression.

The only BH-FDR-significant miRNA, hsa-miR-1292-5p, had no
database-supported interaction with the 114 moderated-test-only audit
mRNA candidates.

Accordingly, the miRNA–mRNA integration should be interpreted as
exploratory. Database support and inverse expression provide
complementary evidence, but no interaction satisfied both criteria
simultaneously.
